# Muon vs SGD：分阶段线性网络

两层线性网络 $\hat y = W_2 W_1 x$，目标 $W_2 W_1 \approx A$。

**目标分解** 对 $A$ 做 SVD：$A = U \Sigma V^\top$，定义
$$W_1^* = \sqrt{\Sigma}\, V^\top, \quad W_2^* = U \sqrt{\Sigma}$$
则 $W_2^* W_1^* = A$。Phase 1 只训练 $W_1$，以 $\|W_1 - W_1^*\|$ 为损失；Phase 2 只训练 $W_2$，以合成误差 $\|W_2 W_1 - A\|$ 为损失。

**分阶段训练**
- Phase 1（比较 SGD 与 Muon）：冻结 $W_2$，用 SGD 或 Muon 训 $W_1$，最小化 $\|W_1 - W_1^*\|^2 / \|A\|^2$
- Phase 2（固定 SGD）：冻结 Phase 1 得到的 $W_1$，重初始化 $W_2$，仅用 SGD 最小化 $\|W_2 W_1 - A\|^2 / \|A\|^2$，直到 $\|W_2 W_1 - A\| / \|A\| \le \text{THRESHOLD}$

Muon：与 SGD 相同 lr；每步对当前梯度做 NS 正交化，更新量范数为 lr $\|g\|$。

In [ ]:
import math

import torch

torch.set_default_dtype(torch.float64)

D = 64
PHASE1_LR = 2e-1
PHASE2_LR = 2e-2
SEED = 0
THRESHOLD = 0.05
PHASE1_STEPS = 600
PHASE2_MAX_STEPS = 20000

In [ ]:
def zeropower_via_newtonschulz5(G, steps=3, eps=1e-7):
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X = X / (X.norm() + eps)
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.to(dtype=G.dtype)


def muon_direction(g, eps=1e-7):
    direction = zeropower_via_newtonschulz5(g)
    return direction * (g.norm() / (direction.norm() + eps))


class Muon(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3):
        super().__init__(params, dict(lr=lr))

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                g = p.grad
                if g is None:
                    continue
                p.data.add_(muon_direction(g), alpha=-lr)

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_target(d, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.randn(d, d, generator=gen) / math.sqrt(d)


def svd_factors(A):
    u, s, vh = torch.linalg.svd(A.cpu(), full_matrices=False)
    sqrt_s = torch.sqrt(s)
    w1_star = (torch.diag(sqrt_s) @ vh).cuda()
    w2_star = (u @ torch.diag(sqrt_s)).cuda()
    return w1_star, w2_star


def rel_dist(M, target, scale):
    return (M - target).norm().item() / (scale + 1e-12)


def compose_loss(W2, W1, A, norm_A):
    M = W2 @ W1 - A
    return (M ** 2).sum() / (norm_A**2)


def steps_to(values, threshold):
    for i, v in enumerate(values):
        if v <= threshold:
            return i
    return None


def rand_matrix(d, seed):
    state = torch.cuda.get_rng_state()
    torch.cuda.manual_seed(seed)
    W = torch.randn(d, d, device="cuda") / math.sqrt(d)
    torch.cuda.set_rng_state(state)
    return W


def w1_spectrum_flatness(W1):
    s = torch.linalg.svdvals(W1)
    return (s.pow(2).sum() / s[0].pow(2)).item() / D

In [ ]:
def train_staged(
    A,
    w1_star,
    w2_star,
    phase1_optimizer_class,
    phase1_steps=PHASE1_STEPS,
    phase2_max_steps=PHASE2_MAX_STEPS,
    phase2_threshold=THRESHOLD,
    phase2_lr=PHASE2_LR,
    seed=SEED,
):
    set_seed(seed)
    W1 = rand_matrix(D, seed)
    norm_A = A.norm().item()
    w1_star = w1_star.detach()

    w1_dist = []

    W1.requires_grad_(True)
    opt1 = phase1_optimizer_class([W1], lr=PHASE1_LR)
    for step in range(phase1_steps + 1):
        w1_dist.append(rel_dist(W1.detach(), w1_star, norm_A))
        if step == phase1_steps:
            break
        loss = ((W1 - w1_star) ** 2).sum() / (norm_A**2)
        opt1.zero_grad(set_to_none=True)
        loss.backward()
        opt1.step()

    W1_frozen = W1.detach().clone()

    W2 = rand_matrix(D, seed + 1)
    W2.requires_grad_(True)
    opt2 = torch.optim.SGD([W2], lr=phase2_lr)
    tot_dist, w2_dist = [], []
    phase2_steps = 0
    for step in range(phase2_max_steps + 1):
        W2d = W2.detach()
        dist = rel_dist(W2d @ W1_frozen, A, norm_A)
        tot_dist.append(dist)
        w2_dist.append(rel_dist(W2d, w2_star, norm_A))
        phase2_steps = step
        if dist <= phase2_threshold:
            break
        if step == phase2_max_steps:
            break
        loss = compose_loss(W2, W1_frozen, A, norm_A)
        opt2.zero_grad(set_to_none=True)
        loss.backward()
        opt2.step()

    return dict(
        tot_dist=tot_dist,
        w1_dist=w1_dist,
        w2_dist=w2_dist,
        W1=W1_frozen,
        W2=W2.detach(),
        phase1_steps=phase1_steps,
        phase2_steps=phase2_steps,
    )


def run_staged_experiment(A, w1_star, w2_star):
    print("\n" + "=" * 72)
    print(f"Staged training: phase1 ({PHASE1_STEPS} steps), phase2 (threshold {THRESHOLD})")
    print(f"  phase1 lr={PHASE1_LR}, loss = ||W1-W1*||^2/||A||^2")
    print(f"  phase2 lr={PHASE2_LR}, loss = ||W2 W1 - A||^2/||A||^2")
    print("=" * 72)
    sgd = train_staged(A, w1_star, w2_star, torch.optim.SGD)
    muon = train_staged(A, w1_star, w2_star, Muon)
    norm_A = A.norm().item()

    for label, run in [("SGD W1 -> SGD W2", sgd), ("Muon W1 -> SGD W2", muon)]:
        p1 = run["phase1_steps"]
        print(f"\n  [{label}]")
        print(f"    phase1 step {p1}: ||W1-W1*||={run['w1_dist'][-1]:.4f}")
        print(f"    phase2 step {p1 + run['phase2_steps']}: ||W1-W1*||={rel_dist(run['W1'], w1_star, norm_A):.4f}, "
              f"||W2-W2*||={run['w2_dist'][-1]:.4f}, ||W2 W1 - A||/||A||={run['tot_dist'][-1]:.6f}, "
              f"steps={run['phase2_steps']}")
        p2_hit = steps_to(run["tot_dist"], THRESHOLD)
        total = p1 + p2_hit if p2_hit is not None else None
        print(f"    steps@{THRESHOLD} total={total}")

    sgd_p2 = steps_to(sgd["tot_dist"], THRESHOLD)
    muon_p2 = steps_to(muon["tot_dist"], THRESHOLD)
    delta = sgd_p2 - muon_p2 if sgd_p2 and muon_p2 else None
    print(f"\n  delta={delta} (positive => Muon faster)")

    print(f"\n  Phase1 end W1 spectrum flatness (higher => flatter):")
    for label, run in [("SGD W1", sgd), ("Muon W1", muon)]:
        flat = w1_spectrum_flatness(run["W1"])
        print(f"    {label}: {flat:.4f}")
    flat_s = w1_spectrum_flatness(sgd["W1"])
    flat_m = w1_spectrum_flatness(muon["W1"])
    print(f"    Muon - SGD: {flat_m - flat_s:+.4f}")

    return sgd, muon

In [ ]:
set_seed(SEED)
A = make_target(D, seed=SEED).cuda()
w1_star, w2_star = svd_factors(A)
sgd_run, muon_run = run_staged_experiment(A, w1_star, w2_star)

## 小结

Muon Phase1 离 $W_1^*$ 更远，但 $W_1$ 谱更利于 Phase2。净效果：Phase2 步数可能更少，整体优化过程仍可能更快（`delta > 0`）。

为什么 Muon 反而在 Phase 2 更快？

Phase 2 等价于固定编码 $W_1$、学线性探测 $W_2$ 使 $W_2 W_1 \approx A$。令 $E_t = W_{2,t}W_1 - A$，则
$$L(W_2)=\tfrac12\|W_2 W_1 - A\|_F^2,\qquad \nabla_{W_2}L=(W_2 W_1 - A)W_1^\top$$
SGD 更新 $W_{2,t+1}=W_{2,t}-\eta(W_{2,t}W_1-A)W_1^\top$，误差递推为
$$E_{t+1}=E_t(I-\eta W_1^\top W_1)$$
在 $W_1^\top W_1$ 的特征基下，$E_t$ 的第 $i$ 个奇异值 $s_i(E_t)$ 满足 $s_i(E_{t+1})=|1-\eta\sigma_i^2|\,s_i(E_t)$，其中 $\sigma_i$ 是 $W_1$ 的第 $i$ 个奇异值。取 $\eta=\alpha/\sigma_{\max}(W_1)^2$ 时，最慢收缩因子为 $1-\alpha/\kappa(W_1)^2$。

因此，$W_1$ 越接近正交（谱越平、条件数越小），Phase 2 的优化越快。

上文直接用 $W_2 W_1 - A$ 构造损失；实践中更常见的做法是给定样本 $x$，计算输出 $y=W_2 W_1 x$，再比较 $y$ 与目标 $Ax$。

可以证明，$W_1$ 越接近正交，中间层表征 $h=W_1 x$ 经线性探测恢复原始输入 $x$ 就越容易——这与前面说的学习 $Ax$ 更容易是同样的道理。恢复 $x$ 时学探测 $B$ 使 $B W_1 \approx I$，损失为
$$L_x(B)=\tfrac12\|B W_1 - I\|_F^2$$
恢复 $Ax$ 时只需把目标矩阵换成 $A$：
$$L_A(B)=\tfrac12\|B W_1 - A\|_F^2$$
令 $E_t = B_t W_1 - M$（$M=I$ 或 $M=A$），则仍有 $E_{t+1} = E_t(I - \eta W_1^\top W_1)$，收敛快慢同样只由 $W_1$ 的奇异值决定。

## 对 Muon 的理解

综上所述，Muon 的优势可以这样理解：上游每次更新同时承担两种角色——一是沿负梯度方向降低当前损失，让训练继续；二是像临时的残差连接，尽量把输入信息原样编码进输出、传递给下游。因为下游究竟更需要 $x$ 还是 $W_1 x$ 事先并不清楚，用正交变换编码 $x$ 是最利于下游线性恢复 $x$ 的做法。Muon 的权衡，正是把梯度正交化后用作更新方向：对 Phase 1 这个「子问题」而言，它不再是该损失下的最速下降；但由此得到的 $W_1$ 为下游提供了更好的表征，Phase 2 反而更容易，而 Phase 2 优化得更好，反过来又可以给 Phase 1 提供更容易的「子问题」，弥补了在 Phase 1 里 Muon 「解题能力」 不如 SGD 的弱点，最终让 Muon 的整体优化反而更快。

## 扩展：带显式 Jacobian 的黑盒下游

对于更一般的下游模块，假设 Phase 1 仍然冻结 $W_1$，但 Phase 2 训练一个可微的黑盒映射
$$
F_\theta(W_1) \approx A.
$$
这里 $F_\theta$ 可以是非线性的，也不需要暴露简单的矩阵分解形式。我们唯一需要的对象，是展平后的预测对下游参数的 Jacobian：
$$
e_t=\operatorname{vec}(F_{\theta_t}(W_1)-A),
\qquad
J_t=\frac{\partial e_t}{\partial \theta_t}.
$$
对于归一化平方损失
$$
L(\theta)=\frac{1}{2\|A\|_F^2}\|e(\theta)\|_2^2,
$$
GD 给出
$$
\theta_{t+1}=\theta_t-\frac{\eta}{\|A\|_F^2}J_t^\top e_t.
$$
一步预测来自一阶 Taylor 展开。在 $\theta_t$ 附近，
$$
e(\theta_{t+1})
\approx
e(\theta_t)+J_t(\theta_{t+1}-\theta_t).
$$
把 GD 更新代入可得
$$
e_{t+1}
\approx
e_t-\frac{\eta}{\|A\|_F^2}J_tJ_t^\top e_t
=
\left(I-\frac{\eta}{\|A\|_F^2}J_tJ_t^\top\right)e_t.
$$
因此，黑盒版本中对应于前面指数衰减公式的对象由 Jacobian Gram 矩阵控制：
$$
K_t=J_tJ_t^\top.
$$
接下来的近似是在估计这一步更新后的误差范数。记
$$
\rho_t=\frac{\eta}{\|A\|_F^2}\frac{e_t^\top K_t e_t}{e_t^\top e_t}.
$$
把一步预测的平方范数展开：
$$
\|e_{t+1}\|_2^2
\approx
\|e_t\|_2^2
-2\frac{\eta}{\|A\|_F^2}e_t^\top K_t e_t
+O(\eta^2).
$$
忽略 $O(\eta^2)$ 项后，
$$
\frac{\|e_{t+1}\|_2}{\|e_t\|_2}
\approx
\sqrt{1-2\rho_t}
\approx
1-\rho_t,
$$
其中最后一步用了小量近似 $\sqrt{1-z}\approx 1-z/2$。再取对数，并使用 $\log(1-\rho_t)\approx-\rho_t$，就得到局部估计：
$$
\log\frac{\|e_{t+1}\|_2}{\|e_t\|_2}
\approx
-\frac{\eta}{\|A\|_F^2}\frac{e_t^\top K_t e_t}{e_t^\top e_t}.
$$
这是一个局部公式：当 GD 步长足够小、一阶 Taylor 展开可靠、且 $O(\eta^2)$ 项可以忽略时，它会比较准确。

多步公式来自把每一步的 log-ratio 估计相加：
$$
\log\frac{\|e_t\|_2}{\|e_0\|_2}
\approx
-\frac{\eta}{\|A\|_F^2}\sum_{s<t}\lambda_{\mathrm{eff},s},
\qquad
\lambda_{\mathrm{eff},s}=\frac{e_s^\top K_s e_s}{e_s^\top e_s}.
$$
两边取指数可得
$$
\|e_t\|_2
\approx
\exp\left(-\frac{\eta}{\|A\|_F^2}\sum_{s<t}\lambda_{\mathrm{eff},s}\right)\|e_0\|_2.
$$
如果 $K_t$ 变化较慢，这个多步近似就可以理解为一个具有时变有效速率的平滑指数衰减。
当下游对参数是线性的时，这会退化为前面用到的精确线性递推。对于非线性黑盒，它是一个局部预测；预测误差反映了 Jacobian 漂移和高阶项的影响。

这也解释了黑盒公式与前面线性探测公式之间的关系。公式里没有显式出现 $W_1^\top W_1$，并不意味着 $W_1$ 不再重要；相反，$W_1$ 现在进入了 Jacobian 内部：
$$
J_t = J_t(W_1)=\frac{\partial\,\operatorname{vec}(F_{\theta_t}(W_1)-A)}{\partial \theta_t}.
$$
改变上游表示会改变黑盒输出对其参数的敏感度，因此也会改变 $K_t=J_t(W_1)J_t(W_1)^\top$。

单线性下游正是这个 Jacobian 视角下最简单的特例。如果
$$
F_\theta(W_1)=BW_1,
$$
其中下游参数是 $\theta=\operatorname{vec}(B)$，那么 Jacobian Gram 算子作用在误差 $E$ 上就是
$$
K(E)=E W_1^\top W_1.
$$
因此，通用 Jacobian 递推会精确退化为代码中归一化损失对应的公式
$$
E_{t+1}=E_t\left(I-\frac{\eta}{\|A\|_F^2}W_1^\top W_1\right).
$$
在线性探测中，$W_1$ 的作用显式表现为 $W_1^\top W_1$；而在非线性黑盒中，同样的作用隐含在 $J_t(W_1)J_t(W_1)^\top$ 的条件性和有效特征值里。


In [ ]:
BB_HIDDEN = 64
BB_LR = 8e-2
BB_MAX_STEPS = 15000
BB_THRESHOLD = 0.05
BB_CHECK_EVERY = 5000


def init_blackbox_params(seed=SEED + 10):
    set_seed(seed)
    W_hidden = (torch.randn(BB_HIDDEN, D, device="cuda") / math.sqrt(D)).requires_grad_(True)
    W_out = (torch.randn(D, BB_HIDDEN, device="cuda") / math.sqrt(BB_HIDDEN)).requires_grad_(True)
    return [W_hidden, W_out]


def blackbox_forward(params, W1):
    W_hidden, W_out = params
    return W_out @ torch.tanh(W_hidden @ W1)


def flatten_params(params):
    return torch.cat([p.reshape(-1) for p in params])


def unflatten_blackbox(theta):
    hidden_size = BB_HIDDEN * D
    W_hidden = theta[:hidden_size].reshape(BB_HIDDEN, D)
    W_out = theta[hidden_size:].reshape(D, BB_HIDDEN)
    return [W_hidden, W_out]


def jacobian_and_error_from_theta(theta_anchor, W1, A):
    theta = theta_anchor.detach().requires_grad_(True)

    def error_from_theta(theta_value):
        return (blackbox_forward(unflatten_blackbox(theta_value), W1) - A).reshape(-1)

    error = error_from_theta(theta).detach()
    jacobian = torch.autograd.functional.jacobian(error_from_theta, theta, vectorize=True).detach()
    return jacobian, error


def jacobian_and_error(params, W1, A):
    return jacobian_and_error_from_theta(flatten_params([p.detach() for p in params]), W1, A)


def train_blackbox_downstream(
    W1,
    A,
    lr=BB_LR,
    max_steps=BB_MAX_STEPS,
    threshold=BB_THRESHOLD,
    check_every=BB_CHECK_EVERY,
    record_every=1,
    with_jacobian_checks=False,
):
    params = init_blackbox_params()
    opt = torch.optim.SGD(params, lr=lr)
    norm_A = A.norm().item()
    curve, logs = [], []
    hits = {BB_THRESHOLD: None}

    for step in range(max_steps + 1):
        prediction = blackbox_forward(params, W1)
        error_matrix = prediction - A
        rel_error = error_matrix.norm().item() / norm_A
        if step % record_every == 0:
            curve.append((step, rel_error))
        for th in hits:
            if hits[th] is None and rel_error <= th:
                hits[th] = step

        if with_jacobian_checks and (step % check_every == 0 or rel_error <= threshold or step == max_steps):
            J, e = jacobian_and_error(params, W1, A)
            K = J @ J.T
            scaled_lr = lr / (norm_A**2)
            pred_next_e = e - scaled_lr * (K @ e)
            pred_next_rel = pred_next_e.norm().item() / norm_A
            lambda_eff = (e @ (K @ e) / (e @ e)).item()
            logs.append(
                dict(
                    step=step,
                    rel_error=rel_error,
                    pred_next_rel=pred_next_rel,
                    actual_next_rel=None,
                    lambda_eff=lambda_eff,
                )
            )

        if rel_error <= threshold or step == max_steps:
            break

        loss = 0.5 * (error_matrix ** 2).sum() / (norm_A**2)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        if logs and logs[-1]["step"] == step:
            with torch.no_grad():
                logs[-1]["actual_next_rel"] = (blackbox_forward(params, W1) - A).norm().item() / norm_A

    return dict(curve=curve, logs=logs, hits=hits, final_step=step, final_error=rel_error)


def run_blackbox_jacobian_experiment():
    print("\n" + "=" * 72)
    print("Black-box downstream: D=64 main Phase-1 W1, predicting GD with J J^T")
    print(f"  hidden={BB_HIDDEN}, lr={BB_LR}, threshold={BB_THRESHOLD}")
    print("=" * 72)

    results = {}
    for label, run in [("SGD-trained W1", sgd_run), ("Muon-trained W1", muon_run)]:
        result = train_blackbox_downstream(run["W1"], A, with_jacobian_checks=True)
        results[label] = result
        one_step_errors = [
            abs(row["actual_next_rel"] - row["pred_next_rel"])
            for row in result["logs"]
            if row["actual_next_rel"] is not None
        ]
        print(f"\n  [{label}]")
        print(f"    phase1 ||W1-W1*||={run['w1_dist'][-1]:.4f}, flatness={w1_spectrum_flatness(run['W1']):.4f}")
        print(f"    hits: {result['hits']}, final step={result['final_step']}, final error={result['final_error']:.6f}")
        for row in result["logs"]:
            print(
                f"    step {row['step']:5d}: rel={row['rel_error']:.6f}, "
                f"pred_next={row['pred_next_rel']:.6f}, actual_next={row['actual_next_rel']}, "
                f"lambda_eff={row['lambda_eff']:.3e}"
            )
        print(
            f"    one-step prediction abs error: mean={sum(one_step_errors) / len(one_step_errors):.3e}, "
            f"max={max(one_step_errors):.3e}"
        )

    s = results["SGD-trained W1"]["hits"][BB_THRESHOLD]
    m = results["Muon-trained W1"]["hits"][BB_THRESHOLD]
    delta = s - m if s is not None and m is not None else None
    print(f"\n  delta@{BB_THRESHOLD}={delta} (positive => Muon-trained W1 faster)")
    return results


blackbox_jacobian_results = run_blackbox_jacobian_experiment()


### 多步 Jacobian 预测

前面的检查是一步预测。为了测试这个公式在多步 GD 中的效果，有两种自然的近似方式：

1. **冻结核预测：** 只在第 0 步计算一次 $K_0=J_0J_0^\top$，然后反复应用
$$
\hat e_{t+1}=\left(I-\frac{\eta}{\|A\|_F^2}K_0\right)\hat e_t.
$$
只有当下游映射对参数是线性的，或者 Jacobian 几乎不变时，这才是精确的。

2. **滚动核预测：** 每隔若干步，在真实轨迹上重新计算 $K_t=J_tJ_t^\top$，再用这个局部核预测下一小段。这可以检验当黑盒 Jacobian 在训练中漂移时，局部 Jacobian 公式是否仍然有预测能力。

下图比较了两种预测和真实 GD 曲线。固定 $K_0$ 的曲线能抓住总体趋势，但长程会漂移；滚动 Jacobian 预测更贴近真实曲线，因为它会不断更新局部线性化。


In [ ]:
import matplotlib.pyplot as plt

ROLLING_INTERVAL = 2500
BLACKBOX_MULTISTEP_FIG = "blackbox_jacobian_multistep_prediction.png"


def train_blackbox_with_snapshots(W1, A, max_steps=BB_MAX_STEPS, threshold=BB_THRESHOLD, rolling_interval=ROLLING_INTERVAL):
    params = init_blackbox_params()
    opt = torch.optim.SGD(params, lr=BB_LR)
    norm_A = A.norm().item()
    actual_errors, snapshots = [], []

    for step in range(max_steps + 1):
        with torch.no_grad():
            rel_error = (blackbox_forward(params, W1) - A).norm().item() / norm_A
            actual_errors.append(rel_error)
        if step % rolling_interval == 0 or rel_error <= threshold or step == max_steps:
            snapshots.append((step, flatten_params(params)))
        if rel_error <= threshold or step == max_steps:
            break

        error_matrix = blackbox_forward(params, W1) - A
        loss = 0.5 * (error_matrix ** 2).sum() / (norm_A**2)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    return torch.tensor(actual_errors), snapshots


def frozen_kernel_prediction(theta_anchor, W1, A, horizon):
    J, e = jacobian_and_error_from_theta(theta_anchor, W1, A)
    K = J @ J.T
    scaled_lr = BB_LR / (A.norm().item() ** 2)
    pred_errors = []
    current_e = e.clone()
    norm_A = A.norm().item()

    for _ in range(horizon + 1):
        pred_errors.append(current_e.norm().item() / norm_A)
        current_e = current_e - scaled_lr * (K @ current_e)

    lambda_eff = (e @ (K @ e) / (e @ e)).item()
    return torch.tensor(pred_errors), lambda_eff


def rolling_kernel_prediction(W1, A, snapshots, total_steps):
    pred_errors = torch.empty(total_steps + 1)
    lambda_eff_by_anchor = []
    for idx, (start, theta_anchor) in enumerate(snapshots[:-1]):
        end = snapshots[idx + 1][0]
        segment_pred, lambda_eff = frozen_kernel_prediction(theta_anchor, W1, A, end - start)
        pred_errors[start : end + 1] = segment_pred
        lambda_eff_by_anchor.append((start, lambda_eff))
    return pred_errors, lambda_eff_by_anchor


def blackbox_multistep_case(label, W1):
    actual, snapshots = train_blackbox_with_snapshots(W1, A)
    total_steps = len(actual) - 1
    frozen_from_start, lambda_eff_0 = frozen_kernel_prediction(snapshots[0][1], W1, A, total_steps)
    rolling, lambda_eff_by_anchor = rolling_kernel_prediction(W1, A, snapshots, total_steps)
    return dict(
        label=label,
        actual=actual.cpu(),
        frozen=frozen_from_start.cpu(),
        rolling=rolling.cpu(),
        lambda_eff_0=lambda_eff_0,
        lambda_eff_by_anchor=lambda_eff_by_anchor,
    )


def run_blackbox_multistep_prediction_plot():
    cases = [
        blackbox_multistep_case("SGD-trained W1", sgd_run["W1"]),
        blackbox_multistep_case("Muon-trained W1", muon_run["W1"]),
    ]

    print("\n" + "=" * 72)
    print("Black-box downstream: multi-step Jacobian prediction")
    print(f"  fixed K0 vs rolling K_t every {ROLLING_INTERVAL} steps")
    print("=" * 72)
    for case in cases:
        frozen_mae = (case["actual"] - case["frozen"]).abs().mean().item()
        rolling_mae = (case["actual"] - case["rolling"]).abs().mean().item()
        print(f"\n  [{case['label']}]")
        print(
            f"    final actual/frozen/rolling = "
            f"{case['actual'][-1]:.6f} / {case['frozen'][-1]:.6f} / {case['rolling'][-1]:.6f}"
        )
        print(f"    mean abs curve error: frozen={frozen_mae:.3e}, rolling={rolling_mae:.3e}")
        print(f"    lambda_eff at step 0: {case['lambda_eff_0']:.3e}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, case in zip(axes, cases):
        steps = range(len(case["actual"]))
        ax.plot(steps, case["actual"], label="actual GD", lw=2.2, color="#243447")
        ax.plot(steps, case["frozen"], label="multi-step prediction, fixed K0", lw=1.8, ls="--", color="#6C63FF")
        ax.plot(
            steps,
            case["rolling"],
            label=f"rolling prediction, K every {ROLLING_INTERVAL} steps",
            lw=1.8,
            ls=":",
            color="#E83E8C",
        )
        ax.set_title(case["label"])
        ax.set_xlabel("downstream GD step")
        ax.grid(True, alpha=0.25)
    axes[0].set_ylabel(r"relative error $\|F_\theta(W_1)-A\|/\|A\|$")
    axes[1].legend(frameon=False, loc="upper right")
    fig.tight_layout()
    fig.savefig(BLACKBOX_MULTISTEP_FIG, dpi=180)
    plt.show()
    return cases


blackbox_multistep_results = run_blackbox_multistep_prediction_plot()


### 黑盒下游的完整训练曲线

上面的 Jacobian 预测图已经包含真实 GD 曲线，但如果只把真实黑盒下游训练曲线画在同一坐标轴上，会更容易比较。这里的 $W_1$ 不是手工构造的：它就是主实验中使用同样 Phase 1 设置得到的 $D=64$ 上游矩阵，分别来自 SGD 和 Muon。


In [ ]:
BLACKBOX_ACTUAL_CURVES_FIG = "blackbox_actual_training_curves.png"


def run_blackbox_actual_curve_plot():
    sgd_result = train_blackbox_downstream(sgd_run["W1"], A, with_jacobian_checks=False)
    muon_result = train_blackbox_downstream(muon_run["W1"], A, with_jacobian_checks=False)

    print("\n" + "=" * 72)
    print("Black-box downstream: actual training curves")
    print("=" * 72)
    print(
        f"  SGD-trained W1:  phase1 dist={sgd_run['w1_dist'][-1]:.4f}, "
        f"flatness={w1_spectrum_flatness(sgd_run['W1']):.4f}, hits={sgd_result['hits']}"
    )
    print(
        f"  Muon-trained W1: phase1 dist={muon_run['w1_dist'][-1]:.4f}, "
        f"flatness={w1_spectrum_flatness(muon_run['W1']):.4f}, hits={muon_result['hits']}"
    )

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    ax.plot([x for x, _ in sgd_result["curve"]], [y for _, y in sgd_result["curve"]], label="SGD-trained W1", lw=2.2, color="#2F80ED")
    ax.plot([x for x, _ in muon_result["curve"]], [y for _, y in muon_result["curve"]], label="Muon-trained W1", lw=2.2, color="#D946EF")
    ax.axhline(BB_THRESHOLD, color="#243447", lw=1.0, ls=":", alpha=0.65)
    ax.set_xlabel("downstream GD step")
    ax.set_ylabel(r"relative error $\|F_\theta(W_1)-A\|/\|A\|$")
    ax.set_title("Black-box downstream actual training curves")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(BLACKBOX_ACTUAL_CURVES_FIG, dpi=180)
    plt.show()
    return {"sgd": sgd_result, "muon": muon_result}


blackbox_actual_curve_results = run_blackbox_actual_curve_plot()
